# Standing on Giants' Shoulders: Transfer Learning

So far, every network you've built started from nothing - random weights, learning everything from scratch. That's not how most real-world computer vision is done.

**Transfer learning** starts from a network someone else already trained on millions of images (ImageNet), keeps everything it already learned about edges, textures, and shapes, and only retrains the final decision-making layer for *your* categories. It's dramatically faster and needs far less data than training from scratch.

In this notebook you'll fine-tune a real pretrained network - **ResNet18** - on a small dataset **you collect yourself**: photos of 3 categories of small physical objects, taken on your phone.

**Before you start:** take 20-30 photos of each of 3 object categories (small everyday objects - not people or faces, to keep this simple and privacy-friendly). Put each category's photos in its own folder, zip the three folders together, and upload the zip when prompted below.


In [ ]:
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms, models
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## Upload and unzip your photos

Expected structure inside the zip:
```
my_photos/
  category_one/
    photo1.jpg
    photo2.jpg
    ...
  category_two/
    ...
  category_three/
    ...
```


In [ ]:
from google.colab import files
import zipfile
import os

uploaded = files.upload()  # choose your zip file when prompted
zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall("my_photos")

# Find the folder that actually contains the category subfolders
# (handles both my_photos/cat1/... and my_photos/my_photos/cat1/... zip layouts)
root = "my_photos"
entries = [e for e in os.listdir(root) if not e.startswith(".")]
if len(entries) == 1 and os.path.isdir(os.path.join(root, entries[0])):
    root = os.path.join(root, entries[0])

categories = sorted(os.listdir(root))
print("Found categories:", categories)


## Load your photos

`ImageFolder` automatically turns "one subfolder per category" into a labeled dataset - the folder name *is* the label. ResNet18 expects 224x224 images, normalized the same way its original ImageNet training data was, so we match that here.


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

dataset = torchvision.datasets.ImageFolder(root, transform=transform)
loader = DataLoader(dataset, batch_size=8, shuffle=True)

print("Total photos:", len(dataset))
print("Classes (in label order):", dataset.classes)


## Load ResNet18, freeze it, replace the head

`weights="DEFAULT"` downloads ResNet18's ImageNet-trained weights. Freezing every existing parameter (`requires_grad = False`) means training will only ever adjust the brand-new final layer we're about to attach - everything ResNet18 already knows about images stays exactly as it was.


In [ ]:
model = models.resnet18(weights="DEFAULT")

for param in model.parameters():
    param.requires_grad = False

num_classes = len(dataset.classes)
model.fc = nn.Linear(model.fc.in_features, num_classes)  # a fresh, trainable head

model = model.to(device)

# Self-check: confirm only the new head is trainable, everything else is frozen.
# A forgotten freeze loop is one of the most common real transfer-learning bugs.
trainable = [name for name, p in model.named_parameters() if p.requires_grad]
frozen_count = sum(1 for p in model.parameters() if not p.requires_grad)
print("Trainable parameters (should just be the new head):", trainable)
print("Frozen parameter tensors:", frozen_count)


## Fine-tune just the head

Same training loop shape as always - the only difference is `model.parameters()` here effectively only updates the head, since everything else is frozen.


In [ ]:
optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = loss_fn(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()

    avg_loss = running_loss / len(dataset)
    accuracy = correct / len(dataset)
    print(f"epoch {epoch+1}/{EPOCHS}  |  loss {avg_loss:.4f}  |  training accuracy {accuracy:.1%}")


### What you just did

With maybe 60-90 total photos - far too few to train a network from scratch - you fine-tuned a real, production-grade image classifier for your own custom categories, in minutes, on a free GPU. This is genuinely how a huge amount of real-world computer vision gets built: start from a strong pretrained model, and only train what's actually new about your specific problem.
